In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from data_prep import prepare_data, load_data
from feature_eng import add_feature_wear


df = load_data("../data/kufar_auto_v2.csv")
df = prepare_data(df)
df = df.drop(columns=['ad_id', 'subject', 'ad_link', 'list_time'])
df = add_feature_wear(df)
df.info()

Отсеяно по неправдоподобной цене: 43 из 44409
Отсеяно по нереальному пробегу: 448 из 44366
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43918 entries, 0 to 43917
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   price_usd   43918 non-null  float64
 1   regdate     43918 non-null  int64  
 2   mileage     43918 non-null  int64  
 3   engine      43918 non-null  object 
 4   capacity    43918 non-null  float64
 5   gearbox     43918 non-null  object 
 6   body_type   43918 non-null  object 
 7   drive       43918 non-null  object 
 8   seats       43918 non-null  int64  
 9   condition   43918 non-null  object 
 10  brand       43918 non-null  object 
 11  model       43918 non-null  object 
 12  generation  43918 non-null  object 
 13  wear        43918 non-null  float64
dtypes: float64(3), int64(3), object(8)
memory usage: 4.7+ MB


In [2]:
df.head(5)

,price_usd,regdate,mileage,engine,capacity,gearbox,body_type,drive,seats,condition,brand,model,generation,wear
0,4500.00,2001,458000,Дизель,2.0,Механика,Минивэн,Передний,7,С пробегом,Citroen,Evasion,Evasion,17615.4
1,23000.00,2011,395000,Бензин,3.6,Автоматическая,Внедорожник,Полный,5,С пробегом,Porsche,Cayenne,II (958),24687.5
2,3700.00,1998,252000,Бензин,1.4,Механика,Хэтчбек,Передний,5,С пробегом,Mercedes-Benz,A-Класс,I (W168),8689.7
3,649.52,1997,350000,Бензин,1.8,Механика,Универсал,Передний,5,С пробегом,Ford,Mondeo,II,11666.7
4,4800.00,1992,400000,Дизель,1.9,Механика,Фургон,Передний,2,С пробегом,Volkswagen,Transporter,T4,11428.6


In [3]:
col_num = df.select_dtypes(include=['number']).columns.tolist()
col_str = df.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"(col_num), Len { len(col_num)}: {col_num}")
print(f"(col_str), Len {len(col_str)}: {col_str}")


(col_num), Len 6: ['price_usd', 'regdate', 'mileage', 'capacity', 'seats', 'wear']
(col_str), Len 8: ['engine', 'gearbox', 'body_type', 'drive', 'condition', 'brand', 'model', 'generation']


In [4]:
df[col_num].agg(['min', 'max', 'mean', 'median']).round(2)

,price_usd,regdate,mileage,capacity,seats,wear
min,500.00,1980.00,0.00,0.00,2.00,0.00
max,99500.00,2026.00,900000.00,4.00,8.00,150000.00
mean,9340.07,2007.34,252154.11,1.84,5.13,14181.59
median,6250.00,2007.00,250231.00,1.80,5.00,13789.75


In [5]:
pd.DataFrame({col: pd.Series(df[col].nunique()) for col in col_str}).fillna('-')


,engine,gearbox,body_type,drive,condition,brand,model,generation
0,7,2,12,4,2,141,1274,942


In [6]:
pd.DataFrame({col: pd.Series(df[col].unique()) for col in col_str}).fillna('-')


,engine,gearbox,body_type,drive,condition,brand,model,generation
0,Дизель,Механика,Минивэн,Передний,С пробегом,Citroen,Evasion,Evasion
1,Бензин,Автоматическая,Внедорожник,Полный,Новый,Porsche,Cayenne,II (958)
2,Бензин (пропан-бутан),-,Хэтчбек,Задний,-,Mercedes-Benz,A-Класс,I (W168)
3,Бензин (гибрид),-,Универсал,Unknown,-,Ford,Mondeo,II
4,Электричество,-,Фургон,-,-,Volkswagen,Transporter,T4
...,...,...,...,...,...,...,...,...
1269,-,-,-,-,-,-,AMG GT,-
1270,-,-,-,-,-,-,SU7,-
1271,-,-,-,-,-,-,Oriental Son (B11),-
1272,-,-,-,-,-,-,Taunus,-


In [7]:
corr_matrix = df[col_num].corr(method='pearson')
corr_matrix['price_usd'].sort_values(ascending=False)

price_usd    1.000000
regdate      0.742787
wear         0.191036
capacity     0.068053
seats        0.053901
mileage     -0.447986
Name: price_usd, dtype: float64

In [8]:
print((df["mileage"] >= 900000).sum())
print(df[df["mileage"] >= 900000][["mileage", "price_usd"]].head(20))

2
       mileage  price_usd
18075   900000    3251.72
30274   900000    3398.04


In [9]:
print((df["price_usd"] <= 500).sum()) 
print(df[df["price_usd"] <= 500][["mileage", "price_usd", "condition", "brand"]].head(20))

304
      mileage  price_usd   condition       brand
164    126879      500.0  С пробегом        Ford
287    231000      500.0  С пробегом        Opel
864    545456      500.0  С пробегом     Peugeot
1220    90000      500.0  С пробегом      Nissan
1362    31005      500.0  С пробегом  LADA (ВАЗ)
1428   390000      500.0  С пробегом        Ford
1455   555555      500.0  С пробегом        Audi
1456      300      500.0  С пробегом        Audi
1491   400000      500.0  С пробегом     Peugeot
1709   500000      500.0  С пробегом    Chrysler
2118   300000      500.0  С пробегом  LADA (ВАЗ)
2424      400      500.0  С пробегом        Fiat
2475   500000      500.0  С пробегом  Volkswagen
2570        1      500.0  С пробегом        Opel
2573   352523      500.0  С пробегом  Volkswagen
2728    90900      500.0  С пробегом    Wartburg
2786    85800      500.0  С пробегом  LADA (ВАЗ)
2839     1000      500.0  С пробегом  Mitsubishi
2840    54540      500.0  С пробегом  LADA (ВАЗ)
2849   400000   